In [2]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input/datasets/karlaayala/athome")

print("Archivos disponibles:\n")

for path in INPUT_ROOT.rglob("*"):
    if path.is_file():
        print(path)

Archivos disponibles:

/kaggle/input/datasets/karlaayala/athome/full_answer_table.npz
/kaggle/input/datasets/karlaayala/athome/athome3/dataset/dev.csv
/kaggle/input/datasets/karlaayala/athome/athome3/dataset/evaluate.py
/kaggle/input/datasets/karlaayala/athome/athome3/dataset/interactor.py
/kaggle/input/datasets/karlaayala/athome/athome3/dataset/questions_pool.txt
/kaggle/input/datasets/karlaayala/athome/athome3/dataset/test1.csv
/kaggle/input/datasets/karlaayala/athome/athome3/dataset/animals_pool.txt


In [3]:
import os
import sys
from pathlib import Path

LOCAL_DIR = Path(
    "/kaggle/input/datasets/karlaayala/athome/"
    "athome3/dataset"
)

if not LOCAL_DIR.exists():
    raise FileNotFoundError(
        f"No existe la carpeta: {LOCAL_DIR}"
    )

sys.path.insert(0, str(LOCAL_DIR))
os.chdir(LOCAL_DIR)

print("Carpeta de trabajo:", os.getcwd())
print("Archivos:")

for path in sorted(LOCAL_DIR.iterdir()):
    print(" -", path.name)

Carpeta de trabajo: /kaggle/input/datasets/karlaayala/athome/athome3/dataset
Archivos:
 - animals_pool.txt
 - dev.csv
 - evaluate.py
 - interactor.py
 - questions_pool.txt
 - test1.csv


In [4]:
for path in LOCAL_DIR.rglob("*"):
    print(path.relative_to(LOCAL_DIR))

dev.csv
evaluate.py
interactor.py
questions_pool.txt
test1.csv
animals_pool.txt


In [6]:
import random
import numpy as np
import pandas as pd
import torch

from interactor import Interactor
from evaluate import evaluate, load_pools

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

animals_pool, questions_pool = load_pools()
print(f'animals_pool size:   {len(animals_pool):>6}  (e.g. {animals_pool[:5]})')
print(f'questions_pool size: {len(questions_pool):>6}  (e.g. {questions_pool[:3]})')

# Sanity probe: create one Interactor and ask two questions about an octopus.
probe = Interactor(gold_animal='octopus', animals_pool=animals_pool, questions_pool=questions_pool)
print("\nask('is it a mammal?')      ->", probe.ask('is it a mammal?'))
print("ask('does it live in water?') ->", probe.ask('does it live in water?'))
print('Queries used:', probe.queries_used, '/', probe.budget)

# The model loads in bfloat16, which a T4 (Turing) cannot accelerate. Convert to
# float16 (T4 has fp16 tensor cores) -> identical answers, several times faster.
import torch
if torch.cuda.is_available() and next(Interactor._model.parameters()).dtype != torch.float16:
    Interactor._model = Interactor._model.half()
    print('oracle model -> float16 (faster on T4)')
     

Device: cuda
animals_pool size:     1472  (e.g. ['lion', 'tiger', 'leopard', 'snow leopard', 'cheetah'])
questions_pool size:    559  (e.g. ['is it a mammal?', 'is it a bird?', 'is it a reptile?'])
  [interactor] loading Qwen/Qwen2.5-3B-Instruct on cuda...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  [interactor] LLM ready.

ask('is it a mammal?')      -> no
ask('does it live in water?') -> yes
Queries used: 2 / 15


In [7]:
from pathlib import Path

matches = list(
    Path("/kaggle/input").rglob(
        "animal_question_table*.npz"
    )
)

print("Archivos encontrados:")

for path in matches:
    print(path)

Archivos encontrados:
/kaggle/input/datasets/karlaayala/3athome/animal_question_table (1).npz


In [8]:
import numpy as np

if not matches:
    raise FileNotFoundError(
        "No se encontró animal_question_table*.npz "
        "dentro de /kaggle/input."
    )

TABLE_PATH = matches[0]

data = np.load(
    TABLE_PATH,
    allow_pickle=False
)

answer_table = data["table"]
cached_animals = data["animals"]
cached_questions = data["questions"]

print("Archivo:", TABLE_PATH)
print("Tabla:", answer_table.shape)
print("Tipo:", answer_table.dtype)
print("Animales:", len(cached_animals))
print("Preguntas:", len(cached_questions))

Archivo: /kaggle/input/datasets/karlaayala/3athome/animal_question_table (1).npz
Tabla: (1472, 559)
Tipo: bool
Animales: 1472
Preguntas: 559


In [9]:
from evaluate import load_pools

animals_pool, questions_pool = load_pools()

assert np.array_equal(
    cached_animals.astype(str),
    np.asarray(animals_pool).astype(str)
), "El orden de los animales no coincide."

assert np.array_equal(
    cached_questions.astype(str),
    np.asarray(questions_pool).astype(str)
), "El orden de las preguntas no coincide."

print("La tabla coincide con los pools oficiales.")

La tabla coincide con los pools oficiales.


In [11]:
answer_table = answer_table.astype(
    np.uint8,
    copy=False
)

In [12]:
class RandomBaseline:
    def __init__(self, animals_pool, questions_pool, seed=0):
        self.animals_pool = animals_pool
        self.questions_pool = questions_pool
        self.rng = random.Random(seed)

    def solve(self, interactor):
        guessed = set()
        while not interactor.is_done():
            cand = self.rng.choice(self.animals_pool)
            while cand in guessed:
                cand = self.rng.choice(self.animals_pool)
            guessed.add(cand)
            interactor.guess(cand)

baseline_results = evaluate(RandomBaseline(animals_pool, questions_pool), 'dev.csv')

  25/150 rows  mean_score=0.0000  (0.0s)
  50/150 rows  mean_score=0.0156  (0.0s)
  75/150 rows  mean_score=0.0219  (0.0s)
  100/150 rows  mean_score=0.0242  (0.0s)
  125/150 rows  mean_score=0.0194  (0.1s)
  150/150 rows  mean_score=0.0161  (0.1s)

  Dataset:       dev.csv
  Mean score:    0.0161
  Solved rate:   2.0%
  Mean queries:  14.89 / 15
  Wall time:     0.1s


In [13]:
# Reference solution (optional, slow to init). Demonstrates precompute + match,
# but uses FIXED, non-adaptive questions -> leaves most of the score on the table.
FIXED_QUESTIONS = [
    'is it a mammal?',
    'is it a bird?',
    'is it a fish?',
    'is it an insect?',
    'does it live in water?',
    'can it fly?',
    'is it a carnivore?',
    'is it bigger than a human?',
    'does it have a backbone?',
    'is it commonly kept as a pet?',
    'does it have legs?',
    'does it lay eggs?',
]

class FixedQuestionsReference:
    def __init__(self, animals_pool, questions_pool, max_animals=None):
        self.animals_pool = animals_pool
        self.questions_pool = set(questions_pool)
        self.fixed = [q for q in FIXED_QUESTIONS if q in self.questions_pool]
        from interactor import Interactor
        cand = animals_pool if max_animals is None else animals_pool[:max_animals]
        self.candidates = cand
        print(f'  [reference] precomputing {len(cand)} x {len(self.fixed)} answer table...')
        self.table = {}
        for i, a in enumerate(cand):
            sim = Interactor(gold_animal=a, animals_pool=self.animals_pool,
                             questions_pool=self.questions_pool, budget=10**9)
            self.table[a] = tuple(1 if sim.ask(q) == 'yes' else 0 for q in self.fixed)
            if (i + 1) % 200 == 0:
                print(f'    {i+1}/{len(cand)}')
        print('  [reference] table ready.')

    def solve(self, interactor):
        obs = []
        for q in self.fixed:
            if interactor.remaining_budget() <= 1:
                break
            obs.append(1 if interactor.ask(q) == 'yes' else 0)
        obs = tuple(obs)
        def agree(a):
            vec = self.table[a]
            return sum(1 for x, y in zip(vec, obs) if x == y)
        ranked = sorted(self.candidates, key=agree, reverse=True)
        for a in ranked:
            if interactor.is_done():
                break
            interactor.guess(a)

# Example (commented out by default — uncomment to run; slow to init):
# ref = FixedQuestionsReference(animals_pool, questions_pool)
# ref_results = evaluate(ref, 'dev.csv')

In [15]:
from pathlib import Path
import numpy as np


class MySolution:
    def __init__(
        self,
        animals_pool,
        questions_pool,
        number_of_questions=9,
    ):
        self.animals_pool = list(animals_pool)
        self.questions_pool = list(questions_pool)
        self.number_of_questions = number_of_questions

        # Buscar automáticamente la tabla dentro de los inputs de Kaggle.
        matches = list(
            Path("/kaggle/input").rglob(
                "animal_question_table*.npz"
            )
        )

        if not matches:
            raise FileNotFoundError(
                "No se encontró animal_question_table*.npz "
                "dentro de /kaggle/input."
            )

        self.table_path = matches[0]

        print("Cargando tabla:", self.table_path)

        with np.load(
            self.table_path,
            allow_pickle=False,
        ) as data:
            raw_table = np.asarray(
                data["table"],
                dtype=np.uint8,
            )

            cached_animals = [
                str(x) for x in data["animals"]
            ]

            cached_questions = [
                str(x) for x in data["questions"]
            ]

        print("Forma original:", raw_table.shape)

        # Reordenar la tabla para que coincida exactamente
        # con animals_pool y questions_pool.
        animal_to_row = {
            animal: index
            for index, animal in enumerate(cached_animals)
        }

        question_to_column = {
            question: index
            for index, question in enumerate(cached_questions)
        }

        missing_animals = [
            animal
            for animal in self.animals_pool
            if animal not in animal_to_row
        ]

        missing_questions = [
            question
            for question in self.questions_pool
            if question not in question_to_column
        ]

        if missing_animals:
            raise ValueError(
                f"Faltan animales en la tabla: "
                f"{missing_animals[:5]}"
            )

        if missing_questions:
            raise ValueError(
                f"Faltan preguntas en la tabla: "
                f"{missing_questions[:5]}"
            )

        row_order = np.asarray(
            [
                animal_to_row[animal]
                for animal in self.animals_pool
            ],
            dtype=np.int32,
        )

        column_order = np.asarray(
            [
                question_to_column[question]
                for question in self.questions_pool
            ],
            dtype=np.int32,
        )

        self.table = raw_table[
            np.ix_(row_order, column_order)
        ]

        expected_shape = (
            len(self.animals_pool),
            len(self.questions_pool),
        )

        if self.table.shape != expected_shape:
            raise ValueError(
                f"Forma incorrecta: {self.table.shape}; "
                f"esperada: {expected_shape}"
            )

        print("Tabla preparada:", self.table.shape)
        print(
            "Preguntas adaptativas:",
            self.number_of_questions,
        )
        print(
            "Intentos de adivinanza reservados:",
            15 - self.number_of_questions,
        )

    def solve(self, interactor):
        number_of_animals = len(self.animals_pool)
        number_of_questions = len(self.questions_pool)

        # Cantidad de respuestas observadas que contradicen
        # la fila de cada animal.
        mismatches = np.zeros(
            number_of_animals,
            dtype=np.int16,
        )

        asked = np.zeros(
            number_of_questions,
            dtype=bool,
        )

        questions_used = 0

        while (
            questions_used < self.number_of_questions
            and not interactor.is_done()
            and interactor.remaining_budget() > 1
        ):
            best_mismatch = int(mismatches.min())

            # Animales que actualmente explican mejor
            # todas las respuestas observadas.
            elite = np.flatnonzero(
                mismatches == best_mismatch
            )

            # Normalmente usamos solo los mejores candidatos.
            # Si quedan muy pocos demasiado pronto, usamos los
            # mejores 32 para evitar depender de un único error.
            if elite.size >= 4:
                focus = elite
            else:
                ranking = np.argsort(
                    mismatches,
                    kind="stable",
                )

                focus = ranking[
                    :min(32, number_of_animals)
                ]

            # Para cada pregunta, contar cuántos candidatos
            # responderían "yes".
            yes_count = self.table[focus].sum(
                axis=0,
                dtype=np.int32,
            )

            no_count = focus.size - yes_count

            # Una pregunta es mejor mientras más equilibrada
            # sea la división yes/no.
            split_score = np.minimum(
                yes_count,
                no_count,
            ).astype(np.int32)

            # No repetir preguntas.
            split_score[asked] = -1

            question_id = int(
                np.argmax(split_score)
            )

            # Ninguna pregunta restante puede separar candidatos.
            if split_score[question_id] <= 0:
                break

            question = self.questions_pool[
                question_id
            ]

            response = interactor.ask(question)

            observed_answer = (
                1 if response == "yes" else 0
            )

            asked[question_id] = True
            questions_used += 1

            # No eliminamos definitivamente animales.
            # Solamente contamos incompatibilidades.
            predicted_answers = self.table[
                :,
                question_id,
            ]

            mismatches += (
                predicted_answers != observed_answer
            )

        # Primero se prueban los animales con menos
        # contradicciones respecto de las respuestas observadas.
        final_ranking = np.argsort(
            mismatches,
            kind="stable",
        )

        for animal_id in final_ranking:
            if interactor.is_done():
                break

            interactor.guess(
                self.animals_pool[int(animal_id)]
            )

In [16]:
solution = MySolution(
    animals_pool,
    questions_pool,
)

dev_results = evaluate(
    solution,
    "dev.csv",
)

print(dev_results)

Cargando tabla: /kaggle/input/datasets/karlaayala/3athome/animal_question_table (1).npz
Forma original: (1472, 559)
Tabla preparada: (1472, 559)
Preguntas adaptativas: 9
Intentos de adivinanza reservados: 6
  25/150 rows  mean_score=0.7496  (24.9s)
  50/150 rows  mean_score=0.7664  (50.0s)
  75/150 rows  mean_score=0.7720  (74.6s)
  100/150 rows  mean_score=0.7752  (99.4s)
  125/150 rows  mean_score=0.7757  (124.8s)
  150/150 rows  mean_score=0.7772  (150.0s)

  Dataset:       dev.csv
  Mean score:    0.7772
  Solved rate:   99.3%
  Mean queries:  10.91 / 15
  Wall time:     150.0s
{'n': 150, 'mean_score': 0.7771999999999999, 'solved_rate': 0.9933333333333333, 'mean_queries': 10.906666666666666, 'per_row':                       gold  queries_used  solved  score
0                 aardwolf            11       1   0.78
1                  abalone            11       1   0.78
2             african lion            12       1   0.76
3                   alpaca            11       1   0.78
4   

In [17]:
test1_results = evaluate(
    solution,
    "test1.csv",
)

print(test1_results)

  25/500 rows  mean_score=0.7784  (26.1s)
  50/500 rows  mean_score=0.7792  (53.2s)
  75/500 rows  mean_score=0.7797  (81.2s)
  100/500 rows  mean_score=0.7802  (108.9s)
  125/500 rows  mean_score=0.7738  (136.4s)
  150/500 rows  mean_score=0.7709  (164.2s)
  175/500 rows  mean_score=0.7719  (191.8s)
  200/500 rows  mean_score=0.7727  (218.6s)
  225/500 rows  mean_score=0.7738  (244.8s)
  250/500 rows  mean_score=0.7713  (272.0s)
  275/500 rows  mean_score=0.7721  (299.2s)
  300/500 rows  mean_score=0.7678  (326.2s)
  325/500 rows  mean_score=0.7686  (353.4s)
  400/500 rows  mean_score=0.7670  (435.6s)
  425/500 rows  mean_score=0.7680  (462.6s)
  450/500 rows  mean_score=0.7686  (489.8s)
  475/500 rows  mean_score=0.7626  (516.8s)
  500/500 rows  mean_score=0.7618  (544.6s)

  Dataset:       test1.csv
  Mean score:    0.7618
  Solved rate:   97.6%
  Mean queries:  11.07 / 15
  Wall time:     544.6s
{'n': 500, 'mean_score': 0.76176, 'solved_rate': 0.976, 'mean_queries': 11.072, 'per_ro

In [ ]:
import os

solution = MySolution(animals_pool, questions_pool)
solution.show_most_similar_questions()
dev_results   = evaluate(solution, 'dev.csv')
test1_results = evaluate(solution, 'test1.csv')

splits = [('dev', dev_results), ('test1', test1_results)]
# test2 is a held-out set; included automatically only if present in the folder.
if os.path.exists('test2.csv'):
    splits.append(('test2', evaluate(solution, 'test2.csv')))

rows = [{
    'split': name, 'n': r['n'], 'mean_score': r['mean_score'],
    'solved_rate': r['solved_rate'], 'mean_queries': r['mean_queries'],
} for name, r in splits]

# FINAL = n-weighted mean over every test split available (test1 [+ test2]).
tests = [r for name, r in splits if name.startswith('test')]
n_test = sum(r['n'] for r in tests)
rows.append({
    'split': 'FINAL',
    'n': n_test,
    'mean_score':   sum(r['mean_score']   * r['n'] for r in tests) / n_test,
    'solved_rate':  sum(r['solved_rate']  * r['n'] for r in tests) / n_test,
    'mean_queries': sum(r['mean_queries'] * r['n'] for r in tests) / n_test,
})
pd.DataFrame(rows)

In [ ]:
unique_ids, duplicate_groups = (
    solution._remove_exact_duplicate_questions()
)

print("Total de preguntas:", len(solution.fixed_questions))
print("Preguntas únicas:", len(unique_ids))
print("Preguntas eliminables:", (
    len(solution.fixed_questions) - len(unique_ids)
))

for group_number, group in enumerate(
    duplicate_groups,
    start=1
):
    print(f"\nGrupo duplicado {group_number}:")

    for question_id in group:
        print(
            question_id,
            solution.fixed_questions[question_id]
        )

In [ ]:
selected_table = solution.table[
    :,
    solution.selected_question_ids
]

unique_signatures, counts = np.unique(
    selected_table,
    axis=0,
    return_counts=True
)

print("Animales:", len(solution.animals_pool))
print("Firmas únicas:", len(unique_signatures))
print("Tamaño promedio de grupo:", counts.mean())
print("Grupo más grande:", counts.max())
print("Grupos de tamaño 1:", np.sum(counts == 1))
print("Grupos de tamaño <= 5:", np.sum(counts <= 5))

animals_in_small_groups = counts[counts <= 5].sum()

print(
    "Animales en grupos de tamaño <= 5:",
    animals_in_small_groups
)

print(
    "Porcentaje potencialmente cubierto por 5 guesses:",
    animals_in_small_groups / len(solution.animals_pool)
)

In [ ]:
for size in range(1, 11):
    number_of_groups = np.sum(counts == size)

    print(
        f"Grupos con {size} animales:",
        number_of_groups
    )

print("Grupos con más de 10:", np.sum(counts > 10))